<a href="https://colab.research.google.com/github/rkk7452/v2_animated_source_matching/blob/main/v2_Animated_PyBDSF_Match_Sources.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Set up

In [ ]:
# Import necessary packages
import numpy as np
import matplotlib.pyplot as plt
import astropy
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization import ZScaleInterval, ImageNormalize
from astropy.coordinates import SkyCoord
import astropy.units as u
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import math

from matplotlib.ticker import MultipleLocator # custom ticks in plots
import colorcet as cc # more colormaps
from scipy.interpolate import griddata # interpolation for TEC background (old)
from scipy.interpolate import RBFInterpolator # interpolation for TEC background
from datetime import datetime
import pytz
from scipy.spatial import cKDTree # for limiting interpolation to concentrated regions

from collections import defaultdict

In [ ]:
# Set up messaging notification function (AI function but helps with efficiently running long notebooks consecutively)
import requests
from google.colab import userdata

WEBHOOK = userdata.get('discord')

def notify(message):
  r = requests.post(
      WEBHOOK,
      json={"content": message}
  )

  if r.status_code == 204:
      print("Notification sent!")
  else:
      print("Error:", r.status_code, r.text)


In [ ]:
# Gives access to data stored on Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
#import vlssr catalog
!pip install -U --pre astroquery
from astropy.table import Table
from astroquery.vizier import Vizier, conf
import time, os

vlssr_cache = "/content/drive/MyDrive/Caltech SRC '26/vlssr_catalog.fits"

def load_vlssr(max_tries=2):
    # 1) Drive cache
    if os.path.exists(vlssr_cache):
        print("Loaded VLSSr from Drive cache.")
        return Table.read(vlssr_cache)

    expected_rows = 92965  # per HEASARC; used to catch silently truncated downloads

    def save_and_return(cat, source):
      if len(cat) != expected_rows:
          print(f"!! {source} returned {len(cat)} rows, expected {expected_rows}; NOT caching.")
          return cat

      # clean up columns FITS can't store
      if '__row' in cat.colnames:
          cat.remove_column('__row')
      for name in cat.colnames:
          if cat[name].dtype == object:
              cat[name] = cat[name].astype(str)

      try:
          cat.write(vlssr_cache, format="fits", overwrite=True)
          print(f"Downloaded {len(cat)} sources from {source}; cached to Drive.")
      except Exception as e:
          print(f"Downloaded {len(cat)} sources from {source}, but caching failed: "
                f"{type(e).__name__}: {e}")
      return cat

    for attempt in range(max_tries):
        # 2) HEASARC (NASA)
        try:
            from astroquery.heasarc import Heasarc
            res = Heasarc.query_tap("SELECT * FROM public.vlssr")
            cat = res.to_table() if hasattr(res, "to_table") else res
            return save_and_return(cat, "HEASARC")
        except Exception as e:
            print(f"HEASARC failed (attempt {attempt+1}): {type(e).__name__}: {e}")

        # 3) VizieR mirrors
        conf.timeout = 120
        Vizier.ROW_LIMIT = -1
        for server in ["vizier.cds.unistra.fr", "vizier.cfa.harvard.edu", "vizier.hia.nrc.ca"]:
            try:
                conf.server = server
                cat = Vizier.get_catalogs("VIII/97")[0]
                return save_and_return(cat, server)
            except Exception as e:
                print(f"{server} failed (attempt {attempt+1}): {type(e).__name__}")
        time.sleep(10)

    print("!! VLSSr download failed everywhere; continuing without it.")
    return None

vlssr_catalog = load_vlssr()
if vlssr_catalog is not None:
    print(vlssr_catalog.colnames)

#Variables to change each run

In [ ]:
#Notebook Variables:
date = "20250508" # Format: "YYYYMMDD"
freq = 73


#Pre Matching Filters:
min_elevation = 5 # anything lower is removed from the dataset---deletes below horizon anomalies and horizon-level contamination
min_flux = 1 #minimum brightness for a source to be kept, Jy/beam


#Matching Thresholds
sep_limit = 5 * u.arcmin # sources can be a max of seplimit arc min apart to be considered a match
MAX_GROUP_RADIUS_DEG = 4.5 / 60  # 4.5 arcmin, independent of sep_limit; for centroid grouping
fluxlimit = 0.2 # 20% limit in flux difference between sources (not being used as of 7/14/2026)


#Post Matching Filters:
min_detections = 20 # minimum times a source has to be found, across images


#TEC Background Cropping Variables
resolution = 100
cutoff = 0.05 # percentile to trim from each tail (outlier removal), for the interpolation grid size
pad_frac = 0.4 # extra margin around trimmed ends, for interpolation grid size
crop_tec = True # whether to crop edges of TEC background to try to reduce edge artifacts, for interpolation grid size
  #If False, the boundary is pad_frac larger than the cutoff percentile.
  #If True, pad_prac is considered for interpolation, but the plotting is cutoff at the cutoff percentile


#Folders
output_folder = f"/content/drive/MyDrive/Caltech SRC '26/Output/{date}/" # where to export plots and data to
deep_int_folder = "/content/drive/MyDrive/Caltech SRC '26/Deep-Integrated-Sources-Catalog/" # where the deep int source CSVs are
short_int_folder = "/content/drive/MyDrive/Caltech SRC '26/Short-Integrated-Sources-Catalog/" # where the short int source CSVs are
gim_folder = "/content/drive/MyDrive/Caltech SRC '26/GIM-Interpolations/" # Where GIM CSVs are stored

#Set up Dataframes

In [ ]:
df = pd.read_csv(os.path.join(output_folder, short_int_folder, date, f"combined_sources_{freq}MHz_{date}.csv"))

df = df.sort_values(by='file_name',ascending=True)
df.insert(6, "epoch", pd.to_datetime(
    df["file_name"].str.extract(r"(\d{8}_\d{6})")[0],
    format="%Y%m%d_%H%M%S",
    utc=True
))

df.insert(1, 'detection_id',
    df["epoch"].dt.strftime("%Y%m%d_%H%M%S_") + str(freq) + "_" + df["source_id"].astype(str)
)

display(df.head())

In [ ]:
#set up

unique_files = df['file_name'].unique()


In [ ]:
###temp###

#Look at elevation data
df = df.sort_values(by='elevation')
display(df.head())

fig, ax = plt.subplots()
x = df['elevation']


bins = np.linspace(-10, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Source Elevation', color='blue')


plt.xlabel("Elevation (deg)")
plt.ylabel("# of Occurances")
plt.xlim(-10,100)
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))
plt.legend(loc='upper right')
plt.title("Source Elevation")
plt.show()

In [ ]:
# set up master catalog and detections df's


master_catalog = pd.read_csv(os.path.join(deep_int_folder,date,f"{freq}MHz_{date}_Deep_Integration_Sources.csv"))

master_catalog.insert(0, 'master_id', range(0, len(master_catalog)))

detections = pd.DataFrame(columns=['master_id', 'source_num', 'ra', 'dec', 'total_flux', 'elevation', 'file_name'])

master_catalog = master_catalog.set_index("master_id")
master_catalog['num_detections'] = 0

master_catalog = master_catalog.rename(columns={'ra': 'long_ra', 'dec': 'long_dec', 'total_flux': 'long_flux'})

display(master_catalog.head())
print(f'Length of Master Catalog: {len(master_catalog)}')

master_coords = SkyCoord(ra=master_catalog['long_ra'].values*u.deg, dec=master_catalog['long_dec'].values*u.deg) # first set to match from


#Pre-Match Filtering

In [ ]:
#Filter main df and master catalog to sources sufficiently above the horizon:

print(len(df))

# drop values <= min_elevation
df = df[df['elevation'] > min_elevation].copy() # Filter short integrated sources
master_catalog = master_catalog[master_catalog['elevation'] > min_elevation].copy() # Filter deep integrated sources

print(len(df))

In [ ]:
#Filter main df and master catalog to sources sufficiently bright enough

print(len(df))

# drop values < min_flux
df = df[df['total_flux'] >= min_flux].copy() # Filter deep integrated sources
master_catalog = master_catalog[master_catalog['long_flux'] >= min_flux].copy() # Filter deep integrated sources

print(len(df))

In [ ]:
# reset SkyCoord objects after filtering
master_coords = SkyCoord(
    ra=master_catalog["long_ra"].values * u.deg,
    dec=master_catalog["long_dec"].values * u.deg,
)

#Match Source Pairs

In [ ]:
# Set up basic matching:
detection_ids = df['detection_id'].values
snap_code, _ = pd.factorize(df['epoch']) # integer snapshot per row

coords_list = SkyCoord(ra=df.ra.values*u.deg, dec=df.dec.values*u.deg)  # no obstime needed
idx_1, idx_2, d2d, _ = astropy.coordinates.search_around_sky(coords_list, coords_list, sep_limit)

#  mask: drop self/reverse pairs and same-snapshot pairs
keep = (idx_1 < idx_2) & (snap_code[idx_1] != snap_code[idx_2])
idx_1, idx_2, dist = idx_1[keep], idx_2[keep], d2d.arcsec[keep]
del d2d

order = np.argsort(dist)
idx_1, idx_2 = idx_1[order], idx_2[order]
print(f"{len(idx_1):,} candidate pairs")

In [ ]:
n = len(df)
node_snapshot = snap_code
node_ra  = np.deg2rad(df['ra'].values)
node_dec = np.deg2rad(df['dec'].values)

In [ ]:
#Set up DSU (Disjoint Set Union)

# Number of unique nodes involved in candidate matches


if (node_snapshot < 0).any() or np.isnan(node_ra).any() or np.isnan(node_dec).any():
    print("!! NaN detected !!")

size = [1] * n
parent = list(range(n))
snapshot_sets = [{int(node_snapshot[k])} for k in range(n)]

# Per-group sum of unit vectors (direction of the sum = group centroid; wraparound-safe)
gx = list(np.cos(node_dec) * np.cos(node_ra))
gy = list(np.cos(node_dec) * np.sin(node_ra))
gz = list(np.sin(node_dec))

n_rejected_snapshot = 0
n_rejected_extent = 0

#optimized DSU from https://medium.com/@dineshpandeyy/disjoint-set-union-dsu-a-deep-dive-with-optimizations-0eba7ef78e54, then iterated with Claude to improve

def find(i):
    if parent[i] != i:
        parent[i] = find(parent[i])
    return parent[i]

#used Claude to help develop centroid distance limitations:
def centroid_sep_deg(a, b):
    # angle between the two groups' centroid directions
    cx = gy[a]*gz[b] - gz[a]*gy[b]
    cy = gz[a]*gx[b] - gx[a]*gz[b]
    cz = gx[a]*gy[b] - gy[a]*gx[b]
    cross = math.sqrt(cx*cx + cy*cy + cz*cz)
    dot = gx[a]*gx[b] + gy[a]*gy[b] + gz[a]*gz[b]
    return math.degrees(math.atan2(cross, dot))


def union(x, y):
    global n_rejected_snapshot, n_rejected_extent
    rootX, rootY = find(x), find(y)

    if rootX == rootY:
        return False

    if not snapshot_sets[rootX].isdisjoint(snapshot_sets[rootY]):
        n_rejected_snapshot += 1
        return False

    if centroid_sep_deg(rootX, rootY) > MAX_GROUP_RADIUS_DEG:
        n_rejected_extent += 1
        return False

    if size[rootX] < size[rootY]:
        big, small = rootY, rootX
    else:
        big, small = rootX, rootY

    parent[small] = big
    snapshot_sets[big].update(snapshot_sets[small])
    gx[big] += gx[small]
    gy[big] += gy[small]
    gz[big] += gz[small]
    size[big] += size[small]
    return True


In [ ]:
# Run DSU:

# Match all the pairs
for a, b in zip(idx_1.tolist(), idx_2.tolist()):
    union(a, b)

print(f"Rejected by snapshot collision: {n_rejected_snapshot}")
print(f"Rejected by extent cap: {n_rejected_extent}")

# find the final group root for each node
roots = [find(k) for k in range(len(df))]

# Group DSU ids by root & map back to original df
groups = defaultdict(list)
for dsu_id, root in enumerate(roots):
  groups[root].append(detection_ids[dsu_id])

root_to_master_id = {root: new_id for new_id, root in enumerate(groups.keys())}
rows = []
for root, detection_list in groups.items():
  master_id = root_to_master_id[root]
  for detection_id in detection_list:
    rows.append((detection_id, master_id))

mapping_df = pd.DataFrame(rows, columns = ['detection_id', 'master_source_id'])

#display(mapping_df.head())


In [ ]:
group_sizes = mapping_df.groupby('master_source_id').size().sort_values(ascending=False)
top_groups = group_sizes.head(20).index

check_df = df.merge(mapping_df, on='detection_id', how='inner')

for gid in top_groups:
    sub = check_df[check_df['master_source_id'] == gid]
    ra_std, dec_std = sub['ra'].std(), sub['dec'].std()
    ra_range = sub['ra'].max() - sub['ra'].min()
    dec_range = sub['dec'].max() - sub['dec'].min()
    print(f"group {gid}: n={len(sub)}  ra_std={ra_std:.4f}  dec_std={dec_std:.4f}  "
          f"ra_range={ra_range:.4f}  dec_range={dec_range:.4f}")

In [ ]:
#analyze the DSU grouping
print(group_sizes.describe())
print((group_sizes == 271).sum(), "groups at full size")
print(len(group_sizes), "groups total")

cent = check_df.groupby('master_source_id').agg(ra=('ra','mean'), dec=('dec','mean'), n=('ra','size'))
centc = SkyCoord(cent.ra.values*u.deg, cent.dec.values*u.deg)
i1, i2, d, _ = astropy.coordinates.search_around_sky(centc, centc, 5*u.arcmin)
m = i1 < i2
pairs = pd.DataFrame({'g1': cent.index[i1[m]], 'g2': cent.index[i2[m]], 'sep_arcmin': d[m].arcmin})
print(len(pairs), "group pairs with centroids within 5'")
display(pairs.sort_values('sep_arcmin').head(20))

In [ ]:
g = check_df.groupby('master_source_id')
d_ra  = (check_df['ra'] - g['ra'].transform('mean')) * np.cos(np.deg2rad(check_df['dec']))
d_dec = check_df['dec'] - g['dec'].transform('mean')
scatter_arcsec = np.hypot(d_ra, d_dec) * 3600
big = g['ra'].transform('size') > 10

plt.hist(scatter_arcsec[big], bins=50, color='lightblue', edgecolor='black')
plt.xlabel("Offset from group centroid (arcsec)")
plt.ylabel("Detections")
plt.title("Distribution of Offsets from the centroid center")
plt.show()

#Calculate TEC from Offsets

In [ ]:
# Get pyGrad2Surf, needed for Harker O' Leary method for integrating the differential TEC
# pyGrad2Surf was developed by Christopher Jordan, adapted to Python from the Harker O' Leary code
!pip install pyGrad2Surf
from pyGrad2Surf.g2s import g2s

In [ ]:
detections = df.merge(mapping_df, on='detection_id').rename(columns={'master_source_id': 'master_id'})
detections['ra'] = detections['ra'] % 360 # plots assume 0-360 degrees

# keep only groups above min_detections threshold
sizes = detections.groupby('master_id')['detection_id'].transform('size')
detections = detections[sizes >= min_detections].copy()

# group centroid via unit vectors (wraparound-safe)
r, d = np.deg2rad(detections['ra']), np.deg2rad(detections['dec'])
detections['_x'] = np.cos(d) * np.cos(r)
detections['_y'] = np.cos(d) * np.sin(r)
detections['_z'] = np.sin(d)
g = detections.groupby('master_id')
x, y, z = g['_x'].median(), g['_y'].median(), g['_z'].median()

master_catalog = pd.DataFrame({
    'long_ra':  np.degrees(np.arctan2(y, x)) % 360,
    'long_dec': np.degrees(np.arctan2(z, np.hypot(x, y))),
    'long_flux': g['total_flux'].mean(),
    'num_detections': g.size(),
})
detections = detections.drop(columns=['_x', '_y', '_z'])

In [ ]:
### Newest Version: Animation + Offset calculations, but separated ###

# This cell is the offset calculation section

"""FYI
combined_offsets is a list
each iteration of the loop, the temporary offset_df is added to the list
after the loop, combined_offsets is converted to the df total_offsets_df

Equation for conversion to STEC based off of Jordan et al. '18
https://arxiv.org/pdf/1707.04978

Equation for STEC to TEC conversion based off of Ja Soon Shim '09 citing Mannucci et al., 1993
https://digitalcommons.usu.edu/cgi/viewcontent.cgi?article=1418&context=etd

"""

#Constants set up for TECU conversion:

e_c = 1.602176634e-19      # Electron charge [C]
m_e = 9.1093837e-31        # Electron mass [kg]
eps_0 = 8.854187817e-12    # Vacuum permittivity [F/m]
nu = freq * 10**6          # Observation frequency [Hz]

iono_shell_height = 450 # the height of the thin-shell ionosphere model (km)
earth_mean_radius = 6371 # the mean radius of the earth (km)

shell_circumference = 2 * np.pi * (iono_shell_height+earth_mean_radius) * 1000 # circumference of ionosphere shell in meters

# Constant factor from Joran et. al
constant_factor = -(1.0 / (8.0 * np.pi**2)) * (e_c**2 / (eps_0 * m_e)) * (1.0 / nu**2)

# Conversion factor from raw electrons/m^2 to TECU
TECU_CONVERSION = 1e16

combined_offsets = [] # empty list, will add to it each iteration of the loop

vtec_data = [] # array for the integrated vtec

 # find the center of the dataset to center the background:
ra_rad = np.radians(master_catalog['long_ra']) # the ra's in radians
ra_c = np.degrees(np.arctan2(np.mean(np.sin(ra_rad)), np.mean(np.cos(ra_rad)))) % 360 # uses unit circle trig to make 0 and 360 equivalent
dec_c = master_catalog['long_dec'].mean()

d_ra_master = ((master_catalog['long_ra'] - ra_c + 180) % 360 - 180) * np.cos(np.radians(dec_c))
d_dec_master = master_catalog['long_dec'] - dec_c

x_radius = max(abs(d_ra_master.quantile(cutoff)), abs(d_ra_master.quantile(1 - cutoff))) # slightly smaller than the calc version, to actually plot
y_radius = max(abs(d_dec_master.quantile(cutoff)), abs(d_dec_master.quantile(1 - cutoff))) # slightly smaller than the calc version, to actually plot

x_radius_calc = x_radius * (1 + pad_frac) # larger area to pull data from
y_radius_calc = y_radius * (1 + pad_frac) # larger area to pull data from

grid_x, grid_y = np.meshgrid(
    np.linspace(-x_radius_calc, x_radius_calc, resolution),
    np.linspace(-y_radius_calc, y_radius_calc, resolution)
)

for i in range(0, len(unique_files)):

  file_to_check = unique_files[i] # the name of the nth unique file name in the data

  # create smaller df's with only the sources in the file we're looking at
  detections_sub = detections[detections['file_name']==file_to_check].copy() # detections log subset

  #new smaller df with detections_sub and master catalog based on master id
  merged_sub = detections_sub.merge(
      master_catalog,
      left_on='master_id',
      right_index=True,
      suffixes=('', '_master')
  )

  if merged_sub.empty:
    continue


  # Get the raw offset in degrees by finding the difference between master and frame positions of sources
  ra_raw_offset = merged_sub['ra'] - merged_sub['long_ra']
  dec_offset_deg = merged_sub['dec'] - merged_sub['long_dec']

  #correct the ra offsets for planar projection errors
  ra_offset_deg = (ra_raw_offset + 180) % 360 - 180 # still in degrees, allowing offsets to wrap around the sides
  ra_offset_deg = ra_offset_deg * np.cos(np.radians(merged_sub['long_dec'])) # correct for curvature at high/low dec

  #convert the offset from degrees to radians
  theta_RA_rad = np.radians(ra_offset_deg)
  theta_Dec_rad = np.radians(dec_offset_deg)

  # Calculate differential STEC gradient (electrons/m^3) using Jordan et al. constant
  d_stec_ra_raw  = theta_RA_rad / constant_factor
  d_stec_dec_raw = theta_Dec_rad / constant_factor

  # convert to TECU/m (TECU = 10e16 electrons/m^2)
  d_stec_ra_tecu = d_stec_ra_raw / TECU_CONVERSION
  d_stec_dec_tecu = d_stec_dec_raw / TECU_CONVERSION

  # convert to TECU/deg (geometric conversion)
  d_stec_ra_tecu = d_stec_ra_tecu * shell_circumference / 360
  d_stec_dec_tecu = d_stec_dec_tecu * shell_circumference / 360


  # find overall magnitude of dSTEC with pythagorean theorem (TECU/deg)
  d_stec_diag_tecu = np.sqrt(d_stec_ra_tecu**2 + d_stec_dec_tecu**2)

  offset_df = pd.DataFrame({
      'master_id': merged_sub['master_id'],
      'ra': merged_sub['ra'],
      'dec': merged_sub['dec'],
      'ra_offset': ra_offset_deg, # needed to draw vectors
      'dec_offset': dec_offset_deg, # needed to draw vectors
      'flux_offset': merged_sub['total_flux'] - merged_sub['long_flux'], # in Jy/beam
      'elevation': merged_sub['elevation'],
      'offset_dir': np.degrees(np.atan2(dec_offset_deg, ra_offset_deg)), # direction in degrees

      # New Ionospheric Gradients in TECU/deg (dSTEC's)
      'd_stec_ra': d_stec_ra_tecu,
      'd_stec_dec': d_stec_dec_tecu,
      'd_stec_diag': d_stec_diag_tecu,

      'file_name': file_to_check
  })

  # this frame's detections relative to field center:
  offset_df['d_ra'] = ((offset_df['ra'] - ra_c + 180) % 360 - 180) * np.cos(np.radians(dec_c))
  offset_df['d_dec'] = offset_df['dec'] - dec_c

  # get the differential VTEC:
  elevation_series = merged_sub['elevation']# Map the elevation from master_catalog using the master_id index
  offset_df['VTEC_d'] = offset_df['d_stec_diag'] * (1 - ((np.cos(np.radians(elevation_series)))/(1+iono_shell_height/earth_mean_radius))**2)**0.5 # Calculate the VTEC using the mapped elevation
  offset_df['VTEC_RA_d'] = offset_df['d_stec_ra'] * (1 - ((np.cos(np.radians(elevation_series)))/(1+iono_shell_height/earth_mean_radius))**2)**0.5 # Calculate the VTEC using the mapped elevation
  offset_df['VTEC_Dec_d'] = offset_df['d_stec_dec'] * (1 - ((np.cos(np.radians(elevation_series)))/(1+iono_shell_height/earth_mean_radius))**2)**0.5 # Calculate the VTEC using the mapped elevation

  #convert dVTEC to VTEC:

  grid_dra = griddata(np.vstack((offset_df['d_ra'], offset_df['d_dec'])).T, offset_df['VTEC_RA_d'],
                                    (grid_x, grid_y), method='linear', fill_value=0)
  grid_ddec = griddata(np.vstack((offset_df['d_ra'], offset_df['d_dec'])).T, offset_df['VTEC_Dec_d'],
                                     (grid_x, grid_y), method='linear', fill_value=0)

  vtec_grid = g2s(grid_x[0, :], grid_y[:, 0], grid_dra, grid_ddec) # use pyGrad2Surf to integrate the differential ra's and dec's

  # Slightly crop edges to reduce edge artifacts:
  keep_x = np.abs(grid_x[0, :]) <= x_radius
  keep_y = np.abs(grid_y[:, 0]) <= y_radius

  if crop_tec:
    vtec_grid_cropped = vtec_grid[np.ix_(keep_y, keep_x)]
    grid_x_cropped = grid_x[np.ix_(keep_y, keep_x)]
    grid_y_cropped = grid_y[np.ix_(keep_y, keep_x)]

  else:
    vtec_grid_cropped = vtec_grid
    grid_x_cropped = grid_x
    grid_y_cropped = grid_y

  vtec_data.append({
    "file_name": file_to_check,
    "VTEC": vtec_grid_cropped, # vtec grid
    "grid_d_ra": grid_dra, # interpolated differential VTEC RA-component values
    "grid_d_dec": grid_ddec, # interpolated differential VTEC RA-component values
    "grid_x": grid_x_cropped,
    "grid_y": grid_y_cropped,
    "grid_x_full": grid_x,
    "grid_y_full": grid_y
  })

  combined_offsets.append(offset_df) # add to combined_offsets

total_offsets_df = pd.concat(combined_offsets, ignore_index=True) # makes it into a df

vtec_df = pd.DataFrame(vtec_data)


In [ ]:
#get a df for the interpolated dVTEC values to send to Ruby
#display(vtec_df.head())

dVTEC_values_ra = vtec_df.at[0,'grid_d_ra']
dVTEC_values_dec = vtec_df.at[0,'grid_d_dec']
grid_x_values = vtec_df.at[0,'grid_x_full'] + ra_c
grid_y_values = vtec_df.at[0,'grid_y_full'] + dec_c

print(len(dVTEC_values_ra))
print(len(dVTEC_values_dec))

print(len(grid_x_values))
print(len(grid_y_values))

sample_dVTEC = pd.DataFrame({
    'x': grid_x_values.flatten(),
    'y': grid_y_values.flatten(),
    'dVTEC_RA': dVTEC_values_ra.flatten(),
    'dVTEC_Dec': dVTEC_values_dec.flatten()
}
)
display(sample_dVTEC.head())
print(len(sample_dVTEC))

sample_dVTEC.to_csv("/content/drive/MyDrive/Caltech SRC '26/Sample_dVTEC_interpolated_frame.csv")

In [ ]:
###temp###

data_series = total_offsets_df['d_stec_diag']


#bins = np.linspace(0, 4, 100) # x min, x max, bins

plt.hist(data_series, alpha=0.5, label='dSTEC Magnitude', color='blue')

plt.xlabel("Distribution of dSTEC Magnitude")
plt.ylabel("# of Occurances")

plt.legend(loc='upper right')
plt.show()

In [ ]:
print([master_catalog[col] for col in master_catalog])

In [ ]:
###temp###
#Make sure source elevation doesn't skew dSTEC too much

x_series = total_offsets_df['elevation']
y_series = total_offsets_df['d_stec_diag']

plt.scatter(x_series, y_series,s=0.5)
plt.title("dSTEC vs. Source Elevation")
plt.xlabel('Source Elevation (Degrees)')
plt.ylabel('Source dSTEC Magnitude (TECU/m)')
plt.show()

#Import GIM data offsets

In [ ]:
gim_df = pd.read_csv(os.path.join(gim_folder,date,f"{date}_{freq}MHz_GIM_TEC.csv")) # import the csv

gim_df[['Date', 'Time']] = gim_df['DateTime'].str.split(r'[ +]', expand=True, n=2)[[0, 1]]

gim_df['Date'] = gim_df['Date'].str.replace(r"[-]", "", regex=True)
gim_df['Time'] = gim_df['Time'].str.replace(r"[:]", "", regex=True)

display(gim_df.head())

# double check it's the correct date of data:
for i in range(0, len(gim_df['Date'])):
  if gim_df['Date'][i] != date:
    print("Error. Wrong Date")

print("Check Complete.")

#Animations

In [ ]:
# Part 2: The animation part, using the offset df created above

#without background

#set up

fig, ax = plt.subplots(figsize = (16,9), facecolor = 'black', layout='constrained') # black outside background
base_q = ax.quiver([0], [0], [0], [0], [0], cmap=cc.cm['CET_C7'])
base_q.set_clim(-180, 180)

cbar = fig.colorbar(base_q)
cbar.set_label('Offset Direction (Degrees)', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

ax.clear()

#animation loop
def animate (frame_index):
  ax.clear()

  #re apply stylings that are cleared by ax.clear()
  ax.set_facecolor('black') # black background inside the plot
  ax.set_xlabel("RA", color='white', fontsize=12) # label axis
  ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
  ax.tick_params(colors='white') # white ticks
  ax.set_xlim(0, 360) # range of x axis in degrees (RA)
  ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)

  file_to_check = unique_files[frame_index]
  ax.set_title(f"Vector Offsets for {file_to_check}", color='white', fontsize=14) # title plot

  frame_df = total_offsets_df[total_offsets_df['file_name'] == file_to_check]
  if frame_df.empty:
        print(f"Skipping Frame: {frame_index}. File_name: {file_to_check}")
        return
  frame_df = frame_df.dropna(subset=['ra', 'dec', 'ra_offset', 'dec_offset']).copy() # remove any NaN values if they exist for any reason

  #draw vectors:

  #magnitudes = np.hypot(frame_df['ra_offset'], frame_df['dec_offset']) # Option 1: color is based off of arrow magnitude
  magnitudes = frame_df['offset_dir'] # Option 2: color is based off of arrow direction

  q = ax.quiver(
      frame_df['ra'],
      frame_df['dec'],
      frame_df['ra_offset'],
      frame_df['dec_offset'],
      magnitudes,
      angles='xy',
      scale_units='xy',
      scale=0.006 ,      # smaller num is larger arrow length
      width=0.002,      # larger num is wider arrow shaft
      headwidth=5,
      headlength=5,
      cmap=cc.cm['CET_C7'])

ani = FuncAnimation(fig, animate, frames=len(unique_files))
ani.save(os.path.join(output_folder, f"{date}_{freq}MHz_Vector_Animation_No_Background.mp4"), writer='ffmpeg', fps=3, dpi=224)

notify(f'{freq}MHz No-Background animation done! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}')


In [ ]:
# Part 2a: The animation part, using the offset df created above

#with background, background is overall TEC magnitude

#set up plot

fig, ax = plt.subplots(figsize=(16, 9), facecolor='black', layout='constrained')

min_VTEC = 0 # min VTEC
max_VTEC = 1 # max VTEC

# Colorbar 1 (for the interpolation (background colors))
mappable_mesh = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=min_VTEC, vmax=max_VTEC),
    cmap='plasma'
)

cbar_mesh = fig.colorbar(mappable_mesh, ax=ax, pad=0.00, shrink=0.6)
cbar_mesh.set_label('Background - TEC Magnitude (TECU)', color='white', fontsize=12)
cbar_mesh.ax.yaxis.set_tick_params(color='white', labelcolor='white')

# Static Colorbar 2: For the Vector Offsets (arrow colors)
mappable_quiver = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=-180, vmax=180),
    cmap=cc.cm['CET_C7']
)
cbar_quiv = fig.colorbar(mappable_quiver, ax=ax, pad=0.02, shrink=0.6)
cbar_quiv.set_label('Vectors - Offset Direction (Deg)', color='white', fontsize=12)
cbar_quiv.ax.yaxis.set_tick_params(color='white', labelcolor='white')



# set up interpolation grid

ra = np.linspace(0, 360, 250)
dec = np.linspace(-90, 90, 250)
ra_grid, dec_grid = np.meshgrid(ra, dec, indexing='xy')

# Flatten the grid into a (62500, 2) array for the interpolator
grid_points = np.column_stack((ra_grid.ravel(), dec_grid.ravel()))

#animation loop
def animate (frame_index):
  ax.clear()

  #re apply stylings that are cleared by ax.clear()
  ax.set_facecolor('black') # black background inside the plot
  ax.set_xlabel("RA", color='white', fontsize=12) # label axis
  ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
  ax.tick_params(colors='white') # white ticks
  ax.set_xlim(0, 360) # range of x axis in degrees (RA)
  ax.set_ylim(-90, 90) # range of y axis in degrees (DEC)

  file_to_check = unique_files[frame_index]

  ax.set_title(f"Offset Vectors & TEC: {file_to_check}", color='white', fontsize=14) # title plot

  frame_df = total_offsets_df[total_offsets_df['file_name'] == file_to_check]

  frame_time = file_to_check.split("-")[3].split("_")[1]

  frame_gim_tec = gim_df.loc[gim_df['Time'] == frame_time, 'TEC'].iloc[0]

  vtec_sub = vtec_df.loc[vtec_df['file_name'] == file_to_check]
  if vtec_sub.empty:
      return
  vtec_row = vtec_sub.iloc[0] # should only be one row in the vtec_sub, but need a series

  if frame_df.empty:
        print(f"Skipping Frame: {frame_index}. File_name: {file_to_check}")
        return

  frame_df = frame_df.dropna(subset=['ra', 'dec', 'VTEC_d']).copy() # remove any NaN values if they exist for any reason

  #find the mean VTEC value
  frame_vtec_mean = vtec_row['VTEC'].mean()

  vtec_offset = frame_gim_tec - frame_vtec_mean # calculated, but didn't add to VTEC in mesh below, b/c makes it too hard to see internal variations. See option below:

  #create TEC background
  mesh = ax.pcolormesh(
      vtec_row['grid_x']+ra_c,
      vtec_row['grid_y']+dec_c,
      vtec_row['VTEC'], # can incorporate vtec_offset as: vtec_row['VTEC'] + vtec_offset
      vmin=min_VTEC,
      vmax=max_VTEC,
      shading='gouraud',
      cmap='plasma',
      zorder=1
  )

  #draw vectors:

  #magnitudes = np.hypot(frame_df['ra_offset'], frame_df['dec_offset']) # Option 1: color is based off of arrow magnitude
  magnitudes = frame_df['offset_dir'] # Option 2: color is based off of arrow direction

  q = ax.quiver(
      frame_df['ra'],
      frame_df['dec'],
      frame_df['ra_offset'],
      frame_df['dec_offset'],
      magnitudes,
      angles='xy',
      scale_units='xy',
      scale=0.006 ,      # smaller num is larger arrow length
      width=0.002,      # larger num is wider arrow shaft
      headwidth=5,
      headlength=5,
      clim=(-180, 180),
      cmap=cc.cm['CET_C7'],
      alpha=0.8,
      zorder=2)

#ani = FuncAnimation(fig, animate, frames=1)
ani = FuncAnimation(fig, animate, frames=len(unique_files))

ani.save(os.path.join(output_folder, f"{date}_{freq}MHz_Vector_Animation_Background_TEC.mp4"), writer='ffmpeg', fps=3, dpi=224)

notify(f'{freq}MHz {date} VTEC Background animation done! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}')


Mini Animation Version

In [ ]:
# Part 2b: The animation part, using the offset df created above

# Mini Square Version!!
#with background, background is overall TEC magnitude

#set up plot

fig, ax = plt.subplots(figsize=(8, 6), facecolor='black', layout='constrained')

#VTEC bounds should be the same, so commented out
#min_VTEC = 0 # min VTEC
#max_VTEC = 1 # max VTEC

# Colorbar 1 (for the interpolation (background colors))
mappable_mesh = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=min_VTEC, vmax=max_VTEC),
    cmap='plasma'
)

cbar_mesh = fig.colorbar(mappable_mesh, ax=ax, pad=0.00, shrink=0.6)
cbar_mesh.set_label('Background - TEC Magnitude (TECU)', color='white', fontsize=12)
cbar_mesh.ax.yaxis.set_tick_params(color='white', labelcolor='white')

# Static Colorbar 2: For the Vector Offsets (arrow colors)
mappable_quiver = plt.cm.ScalarMappable(
    norm=plt.Normalize(vmin=-180, vmax=180),
    cmap=cc.cm['CET_C7']
)
cbar_quiv = fig.colorbar(mappable_quiver, ax=ax, pad=0.02, shrink=0.6)
cbar_quiv.set_label('Vectors - Offset Direction (Deg)', color='white', fontsize=12)
cbar_quiv.ax.yaxis.set_tick_params(color='white', labelcolor='white')



# set up interpolation grid

ra = np.linspace(ra_c - x_radius, ra_c + x_radius, 250)
dec = np.linspace(dec_c - y_radius, dec_c + y_radius, 250)
ra_grid, dec_grid = np.meshgrid(ra, dec, indexing='xy')

# Flatten the grid into a (62500, 2) array for the interpolator
grid_points = np.column_stack((ra_grid.ravel(), dec_grid.ravel()))

#animation loop
def animate (frame_index):
  ax.clear()

  #re apply stylings that are cleared by ax.clear()
  ax.set_facecolor('black') # black background inside the plot
  ax.set_xlabel("RA", color='white', fontsize=12) # label axis
  ax.set_ylabel("Dec", color='white', fontsize=12) # label axis
  ax.tick_params(colors='white') # white ticks
  ax.set_xlim(ra_c - x_radius, ra_c + x_radius) # range of x axis in degrees (RA)
  ax.set_ylim(dec_c - y_radius, dec_c + y_radius) # range of y axis in degrees (DEC)

  file_to_check = unique_files[frame_index]

  ax.set_title(f"Offset Vectors & TEC: {file_to_check}", color='white', fontsize=14) # title plot

  frame_df = total_offsets_df[total_offsets_df['file_name'] == file_to_check]

  frame_time = file_to_check.split("-")[3].split("_")[1]

  frame_gim_tec = gim_df.loc[gim_df['Time'] == frame_time, 'TEC'].iloc[0]

  vtec_sub = vtec_df.loc[vtec_df['file_name'] == file_to_check]
  if vtec_sub.empty:
      return
  vtec_row = vtec_sub.iloc[0] # should only be one row in the vtec_sub, but need a series

  if frame_df.empty:
        print(f"Skipping Frame: {frame_index}. File_name: {file_to_check}")
        return

  frame_df = frame_df.dropna(subset=['ra', 'dec', 'VTEC_d']).copy() # remove any NaN values if they exist for any reason

  #find the mean VTEC value
  frame_vtec_mean = vtec_row['VTEC'].mean()

  vtec_offset = frame_gim_tec - frame_vtec_mean # calculated, but don't add to VTEC in mesh below, b/c makes it too hard to see internal variations

  #create TEC background
  mesh = ax.pcolormesh(
      vtec_row['grid_x']+ra_c,
      vtec_row['grid_y']+dec_c,
      vtec_row['VTEC'],
      vmin=min_VTEC,
      vmax=max_VTEC,
      shading='gouraud',
      cmap='plasma',
      zorder=1
  )

  #draw vectors:

  #magnitudes = np.hypot(frame_df['ra_offset'], frame_df['dec_offset']) # Option 1: color is based off of arrow magnitude
  magnitudes = frame_df['offset_dir'] # Option 2: color is based off of arrow direction

  q = ax.quiver(
      frame_df['ra'],
      frame_df['dec'],
      frame_df['ra_offset'],
      frame_df['dec_offset'],
      magnitudes,
      angles='xy',
      scale_units='xy',
      scale=0.02 ,      # smaller num is larger arrow length
      width=0.004,      # larger num is wider arrow shaft
      headwidth=5,
      headlength=5,
      clim=(-180, 180),
      cmap=cc.cm['CET_C7'],
      alpha=0.8,
      zorder=2)

#ani = FuncAnimation(fig, animate, frames=1)
ani = FuncAnimation(fig, animate, frames=len(unique_files))

ani.save(os.path.join(output_folder, f"{date}_{freq}MHz_Vector_Animation_Background_TEC_Mini.mp4"), writer='ffmpeg', fps=3, dpi=224)

notify(f'{freq}MHz {date} Mini VTEC Background animation done! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}')


In [ ]:
notify(f"Completed {freq}MHz ({date}) Matching & Animations! {datetime.now(pytz.timezone("US/Pacific")).strftime("%b %d, %Y at %H:%M:%S")}")

#OLD Matching

In [ ]:
###   OLD  ###

for item in unique_files:
  file_2 = df[df['file_name']==item]
  matches_df = pd.DataFrame({
    'f1_source_id': master_catalog.index[idx_1],
    'f2_source_id': file_2.iloc[idx_2]['source_id'].values,
    'f2_ra': file_2.iloc[idx_2]['ra'].values,
    'f2_dec': file_2.iloc[idx_2]['dec'].values,
    'f2_flux': file_2.iloc[idx_2]['total_flux'].values,
    'f2_elevation': file_2.iloc[idx_2]['elevation'].values,

    'separation_arcmin': d2d.to(u.arcmin).value
  })


  #rank and choose only the best new source to match a given source in the master catalog:

  matches_df["flux_ratio"] = matches_df["f2_flux"] / master_catalog.loc[matches_df["f1_source_id"], "long_flux"].values #create flux ratio

  matches_df["score"] = (matches_df["separation_arcmin"] / seplimit.to(u.arcmin).value)

  matches_df = (matches_df.sort_values("score").drop_duplicates(subset="f1_source_id", keep="first").drop_duplicates(subset='f2_source_id',keep='first'))
    # only keeps the best scored match on each end (1-1 mapping)

  #update the master catalog:

  for _, match in matches_df.iterrows():
    master_id = match["f1_source_id"]

    n = master_catalog.loc[master_id, "num_detections"]
    master_catalog.loc[master_id, "num_detections"] = n + 1 # update num detections

  #update the detections df:

  new_rows = []# create empty list

  for _, match in matches_df.iterrows(): # loop through each row in the matches_df
    new_rows.append( {
        "master_id": match['f1_source_id'],
        'source_num': match['f2_source_id'],
        'ra': match['f2_ra'],
        'dec': match['f2_dec'],
        'total_flux': match['f2_flux'],
        'elevation': match['f2_elevation'],
        'file_name': item
    })

  #add the new_rows list to the detections df
  detections = pd.concat([detections, pd.DataFrame(new_rows)], ignore_index=True)

  #update master_coords
  master_coords = SkyCoord(
    ra=master_catalog["long_ra"].values * u.deg,
    dec=master_catalog["long_dec"].values * u.deg
  )


In [ ]:
master_catalog = master_catalog.sort_values(by="num_detections", ascending=False)
display(master_catalog)

In [ ]:
###temp###

#Flux distribution

### with matches, not filtered
x = master_catalog['long_flux']
y = df['total_flux']
z = detections['total_flux']

bins = np.linspace(0, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Deep Integration', color='blue')
plt.hist(y, bins, alpha=0.5, label='Short Integration', color='green')
plt.hist(z, bins, alpha=0.3, label='Matches',color='red')


plt.xlabel("Flux")
plt.ylabel("# of Occurances")

plt.legend(loc='upper right')
plt.title(f"{date} {freq} MHz Flux Distribution With Matches (before filtering)")
plt.savefig(os.path.join(output_folder, f"{date}_{freq}MHz_Flux_Distribution_With_Matches_No_Filter.png"))
plt.show()


### without matches, not filtered


x = master_catalog['long_flux']
y = df['total_flux']
z = detections['total_flux']

bins = np.linspace(0, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Deep Integration', color='blue')
plt.hist(y, bins, alpha=0.5, label='Short Integration', color='green')


plt.xlabel("Flux")
plt.ylabel("# of Occurances")

plt.legend(loc='upper right')
plt.title(f"{date} {freq} MHz Flux Distribution Without Matches (before filtering)")
plt.savefig(os.path.join(output_folder, f"{date}_{freq}MHz_Flux_Distribution_Without_Matches_No_Filter.png"))
plt.show()



In [ ]:
#Filter Results

print(len(master_catalog))

master_catalog = master_catalog[master_catalog['num_detections'] >= min_detections].copy()# drop values < min_detections

detections = detections[detections['master_id'].isin(master_catalog.index)].copy() # make sure detections is up to date with changed master catalog

#display(master_catalog.head())
#display(master_catalog.tail())

print(len(master_catalog))

# clean up ra values to make it 0-360 instead of -360 to 0:
master_catalog['long_ra'] = master_catalog['long_ra'] % 360
detections['ra'] = detections['ra'] % 360

In [ ]:
#now...export results!

master_catalog = master_catalog.sort_values(by='num_detections', ascending=False)
display(master_catalog.head())

master_catalog.to_csv(os.path.join(output_folder,f"{date}_{freq}MHz_Master_Catalog.csv"), index = True, index_label = 'master_id')
detections.to_csv(os.path.join(output_folder, f"{date}_{freq}MHz_Detection_Log.csv"), index = False)

In [ ]:
###temp###

### with matches, after filtering
x = master_catalog['long_flux']
y = df['total_flux']
z = detections['total_flux']

bins = np.linspace(0, 100, 100) # x min, x max, bins

plt.hist(x, bins, alpha=0.5, label='Deep Integration', color='blue')
plt.hist(y, bins, alpha=0.5, label='Short Integration', color='green')
plt.hist(z, bins, alpha=0.3, label='Matches',color='red')


plt.xlabel("Flux")
plt.ylabel("# of Occurances")

plt.legend(loc='upper right')
plt.title(f"{date} {freq}MHz Flux Distribution With Matches (after filtering)")
plt.savefig(os.path.join(output_folder, f"{date}_{freq}MHz_Flux_Distribution_With_Matches_Filtered.png"))
plt.show()